# 🔢 Demo: compresión de imágenes con PCA (MNIST)

> Material de respaldo de la sesión de PCA: **no se trabaja en clase**. Ejecútalo de arriba a abajo y lee las observaciones.

**MNIST** tiene 70,000 dígitos escritos a mano. Cada imagen mide 28 × 28 = **784 píxeles** (0 = blanco, 255 = negro). ¿Cuántos de esos 784 números necesitamos realmente para guardar un dígito reconocible?

⚠️ **Descarga:** la primera vez `fetch_openml` descarga MNIST de openml.org (unos 15 MB comprimidos; tarda de segundos a un par de minutos según tu conexión). Después queda guardado en `~/scikit_learn_data` y carga en segundos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA

X_mnist, y_mnist = fetch_openml('mnist_784', as_frame=False, parser='auto', return_X_y=True)

# Usamos las primeras 60,000 imágenes (el conjunto de entrenamiento tradicional)
X_train = X_mnist[:60_000].astype('float32')
y_train = y_mnist[:60_000]
print(f"Imágenes: {X_train.shape[0]:,} | píxeles por imagen: {X_train.shape[1]}")
print(f"Rango de los píxeles: {X_train.min():.0f} a {X_train.max():.0f}")

💡 Aquí **no estandarizamos**: todos los píxeles están en la misma unidad (intensidad de 0 a 255). Estandarizar le daría el mismo peso a un píxel de la orilla que casi siempre está en blanco que a uno del centro, y sólo amplificaría ruido.

In [ ]:
def plot_digitos(imagenes, por_fila=10, titulo=None, ax=None):
    """Dibuja varias imágenes de 28 x 28 en una cuadrícula."""
    ax = ax or plt.gca()
    imagenes = np.asarray(imagenes).reshape(-1, 28, 28)
    n_filas = int(np.ceil(len(imagenes) / por_fila))
    lienzo = np.zeros((n_filas * 28, por_fila * 28))
    for i, img in enumerate(imagenes):
        f, c = divmod(i, por_fila)
        lienzo[f * 28:(f + 1) * 28, c * 28:(c + 1) * 28] = img
    ax.imshow(lienzo, cmap='binary', vmin=0, vmax=255)
    ax.axis('off')
    if titulo:
        ax.set_title(titulo)


plt.figure(figsize=(9, 2))
plot_digitos(X_train[:20], titulo='Primeros 20 dígitos de MNIST')
plt.show()

---
## 1️⃣ ¿Cuántos componentes para cada nivel de varianza?

Ajustamos un PCA con **todos** los componentes y vemos la varianza acumulada.

In [ ]:
pca_completo = PCA(random_state=42).fit(X_train)
acumulada = np.cumsum(pca_completo.explained_variance_ratio_)

umbrales = pd.DataFrame({
    'varianza_conservada': ['50%', '80%', '90%', '95%', '99%'],
    'componentes': [int(np.argmax(acumulada >= u) + 1) for u in (0.50, 0.80, 0.90, 0.95, 0.99)]
})
umbrales['pct_de_784'] = (100 * umbrales['componentes'] / 784).round(1)

fig = px.line(x=np.arange(1, 785), y=acumulada,
              labels={'x': 'Número de componentes', 'y': 'Varianza acumulada'},
              title='MNIST: varianza acumulada según el número de componentes')
fig.add_hline(y=0.95, line_dash='dot', line_color='gray', annotation_text='95%')
fig.update_layout(width=800, height=420, yaxis_tickformat='.0%')
fig.show()
umbrales

✅ **Qué observar:**
- Con sólo **11** componentes ya tenemos el **50%** de la varianza; con **44**, el 80%
- Para el **95%** bastan **154** componentes: el **19.6%** de los 784 píxeles
- La curva sube rapidísimo y luego se aplana: cientos de píxeles (sobre todo los de las orillas) casi no varían

---
## 2️⃣ Comprimir al 95% y reconstruir

`PCA(n_components=0.95)` elige solo el número de componentes. Con `inverse_transform` regresamos a 784 píxeles para ver qué se perdió.

In [ ]:
pca_95 = PCA(n_components=0.95, random_state=42)
X_comprimido = pca_95.fit_transform(X_train)
X_reconstruido = pca_95.inverse_transform(X_comprimido)

n = pca_95.n_components_
mb_original = X_train.nbytes / 1e6
mb_comprimido = (X_comprimido.nbytes + pca_95.components_.nbytes + pca_95.mean_.nbytes) / 1e6
print(f"Componentes: {n} (de 784) → varianza conservada {pca_95.explained_variance_ratio_.sum():.1%}")
print(f"Ratio de compresión por imagen: 784 / {n} = {784 / n:.1f}x")
print(f"Memoria: {mb_original:.0f} MB → {mb_comprimido:.0f} MB (incluye los componentes y la media)")
print(f"Error cuadrático medio por píxel: {((X_train - X_reconstruido) ** 2).mean():.1f}  (escala 0-255)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 4))
plot_digitos(X_train[::3000][:20], titulo='Originales (784 píxeles)', ax=axes[0])
plot_digitos(X_reconstruido[::3000][:20], titulo=f'Reconstruidos con {n} componentes (95%)', ax=axes[1])
plt.tight_layout()
plt.show()

✅ **Qué observar:**
- Con **154** números por imagen en lugar de 784 (**5.1×** menos) los dígitos siguen perfectamente legibles; sólo se ven un poco borrosos
- Toda la base pasa de **188 MB** a unos **37 MB**
- 💡 Esto es compresión **con pérdida**, como un JPG: no recuperas la imagen exacta, pero sí lo esencial

---
## 3️⃣ Reconstrucción progresiva y "eigen-dígitos"

¿Cómo se ve un mismo dígito con más y más componentes? Y, ¿qué "aprendió" cada componente?

In [ ]:
digito = X_train[[0]]
scores_digito = pca_completo.transform(digito)
niveles = [1, 5, 11, 44, 87, 154, 331]

fig, axes = plt.subplots(1, len(niveles) + 1, figsize=(14, 2.4))
axes[0].imshow(digito.reshape(28, 28), cmap='binary')
axes[0].set_title('Original\n784 px', fontsize=9)
for ax, k in zip(axes[1:], niveles):
    rec = scores_digito[:, :k] @ pca_completo.components_[:k] + pca_completo.mean_
    ax.imshow(rec.reshape(28, 28), cmap='binary', vmin=0, vmax=255)
    ax.set_title(f'{k} comp.\n({acumulada[k - 1]:.0%} var.)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.show()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.3))
for i, ax in enumerate(axes.flat):
    ax.imshow(pca_completo.components_[i].reshape(28, 28), cmap='RdBu_r')
    ax.set_title(f'PC{i + 1}', fontsize=9)
    ax.axis('off')
plt.suptitle('Los primeros 16 componentes ("eigen-dígitos"): rojo = suma tinta, azul = resta')
plt.show()

✅ **Qué observar:**
- Con **1** componente sólo hay una mancha genérica; con **11** (50%) ya se adivina el dígito; desde **44** (80%) se lee claramente
- Cada componente es una **plantilla** de 784 píxeles. Un dígito comprimido es "un poco de la plantilla 1 + un poco de la 2 + …"
- Los primeros componentes capturan formas globales (un "0" contra un "1"); los siguientes, trazos cada vez más finos

## 📝 En resumen
1. PCA comprime imágenes guardando los **scores** y unas cuantas **plantillas** (componentes)
2. El número de componentes controla el intercambio **tamaño ↔ calidad**
3. Con imágenes (misma unidad en todos los píxeles) no hace falta estandarizar